# 01 · EDA del dataset municipal curated

Auditoría inicial del Parquet integrado: estructura, cobertura, periodos, distribuciones y valores extremos. Este notebook no modifica los datos de entrada.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 150)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'curated' / 'municipios.parquet'
REPORTS_DIR = ROOT / 'outputs' / 'reports'
FIGURES_DIR = ROOT / 'outputs' / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.is_file():
    raise FileNotFoundError(f'No existe el dataset curated: {DATA_PATH}')

df = pd.read_parquet(DATA_PATH)
print(f'Filas: {len(df):,} | Columnas: {df.shape[1]}')
df.head()

## Integridad de las claves

In [ ]:
integridad = pd.Series({
    'filas': len(df),
    'municipios_distintos': df['codigo_ine'].nunique(dropna=True),
    'codigos_duplicados': df.duplicated('codigo_ine').sum(),
    'codigos_nulos': df['codigo_ine'].isna().sum(),
    'municipios_nulos': df['municipio'].isna().sum(),
    'provincias_nulas': df['provincia'].isna().sum(),
})
display(integridad.to_frame('valor'))

if integridad[['codigos_duplicados', 'codigos_nulos']].sum() != 0:
    raise ValueError('El universo municipal no tiene una clave única válida')

## Cobertura y valores nulos

In [ ]:
nulos = (
    pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'nulos': df.isna().sum(),
        'porcentaje_nulos': df.isna().mean().mul(100).round(2),
        'valores_distintos': df.nunique(dropna=True),
    })
    .sort_values(['porcentaje_nulos', 'nulos'], ascending=False)
)
nulos.to_csv(REPORTS_DIR / 'eda_nulos.csv', encoding='utf-8', index_label='variable')
display(nulos.query('nulos > 0'))

In [ ]:
variables_cobertura = {
    'edad': 'edad_media_2025',
    'renta': 'renta_neta_media_persona_eur',
    'catastro': 'inmuebles_residenciales',
    'paro': 'paro_por_1000_habitantes',
    'viviendas': 'viviendas_totales',
    'alquiler_colectiva': 'alquiler_precio_colectiva_mediana',
    'alquiler_unifamiliar': 'alquiler_precio_unifamiliar_mediana',
    'criminalidad': 'total_infracciones',
}
cobertura = pd.DataFrame([
    {
        'fuente': fuente,
        'municipios': int(df[columna].notna().sum()),
        'cobertura_pct': round(df[columna].notna().mean() * 100, 2),
    }
    for fuente, columna in variables_cobertura.items()
]).sort_values('cobertura_pct', ascending=False)
display(cobertura)

## Periodos y estadísticos descriptivos

In [ ]:
columnas_periodo = [c for c in df.columns if c.startswith('anio') or c.startswith('periodo')]
for columna in columnas_periodo:
    print(f'\n{columna}')
    display(df[columna].value_counts(dropna=False).sort_index().to_frame('municipios'))

descripcion = df.select_dtypes(include=np.number).describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
descripcion.to_csv(REPORTS_DIR / 'eda_descriptivos.csv', encoding='utf-8')
display(descripcion)

## Asimetría y correlaciones iniciales

In [ ]:
numericas = df.select_dtypes(include=np.number)
asimetria = numericas.skew().sort_values(key=abs, ascending=False).to_frame('asimetria')
display(asimetria.head(30))

variables_revision = [
    'densidad_poblacion_hab_km2',
    'porcentaje_poblacion_menor_18',
    'porcentaje_poblacion_65_mas',
    'renta_neta_media_persona_eur',
    'tamanio_medio_hogar',
    'saldo_interior_por_1000_habitantes',
    'viviendas_turisticas_por_1000_habitantes',
    'paro_por_1000_habitantes',
    'distancia_costa_km',
]
corr = df[variables_revision].corr(method='spearman')
fig, ax = plt.subplots(figsize=(10, 8))
imagen = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.index)
fig.colorbar(imagen, ax=ax, label='Correlación de Spearman')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlaciones_iniciales.png', dpi=160, bbox_inches='tight')
plt.show()

## Decisiones pendientes

- Definir si el modelo principal cubrirá toda España o solo municipios con Catastro estatal.
- No imputar automáticamente fuentes con ausencias estructurales.
- Revisar los municipios de población muy reducida antes de tratar los extremos.
- Criminalidad, alquiler y viviendas censales se estudiarán como modelos o análisis secundarios.